Indexing是一种数据结构，允许用户从Indexing中快速检索感兴趣的数据，在LlamaIndex中，index通常可以由一系列的Document构建出来，然后再构建出一个QueryEngine或者ChatEngine。用户就可以基于这些Engine快速检索出感兴趣的数据，在index内部，则会以Node的形式保存数据，Node则是Document拆分出来的一个片段，另外，index也会暴漏一个Retrieve接口。进一步支持文件检索，通常用的最多就是VectorStoreIndex，存储资源语言处理后的向量数据

文本-------切片（Document）---Node-----Embedding（模型）----向量-----Vertor Store

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from Config.load_key import open_key
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain_community.embeddings import DashScopeEmbeddings

# from llama_index.embeddings.dashscope import DashScopeEmbedding, DashScopeTextEmbeddingModels, \
#     DashScopeTextEmbeddingType
# #使用DashScope的Embedding向量化模型
# embedder = DashScopeEmbedding(
#     model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V2,
#     text_type=DashScopeTextEmbeddingType.TEXT_TYPE_DOCUMENT,
#     api_key=open_key
# )
# ##将中文转成向量值，不同的模型解析出来的向量值也不同
# result = embedder.get_text_embedding_batch(text_to_embedding)

documents = SimpleDirectoryReader('./Source').load_data()
##初始化通义千问的Embedding模型
#这里将langchain针对DashScope实现的DashScopeEmbedding封装成LlamaIndex可以接受的 langchainEmbedding
embed_model = LangchainEmbedding(
    DashScopeEmbeddings(
        dashscope_api_key=open_key,
        model="text-embedding-v1",
    )
)

##文本转成向量库  storage_context不传存的就是内存里面
index = VectorStoreIndex.from_documents(documents=documents, embed_model=embed_model)

retriever = index.as_retriever()
response = retriever.retrieve('怎么退款')
#检索出跟提示词相关的文本数据
for chunk in response:
    print(chunk.node.text)



Indexing构建索引之后，也支持持久化的保存，下次使用的时候可以直接在持久化获取

In [ ]:
from llama_index.core.storage.storage_context import StorageContext
from llama_index.core import load_index_from_storage

##将向量数据保存到对应目录下
index.storage_context.persist(persist_dir='./indexStore')

##创建一个存储上下文，告诉LLamaindex从磁盘里那向量数据
storage_context = StorageContext.from_defaults(persist_dir='./indexStore')
##根据storage_context重新加载
new_index = load_index_from_storage(storage_context=storage_context, embed_model=embed_model)

new_retriever = new_index.as_retriever()
new_response = new_retriever.retrieve('怎么退款')
for chunk in new_response:
    print(chunk.node.text)